### Пятый чекпойнт посвящен использованию более сложных моделей и архитектур в задаче вашего годового проекта.

Трек ML:

От вас ожидается улучшение решения путем:
Использования нелинейных моделей ML (деревья, леса и бустинги)
Работа с признаками - придумать новые признаки, удалить лишние, использовать кластеризацию для получения дополнительных признаков и так далее (feature-engineering)

В результате экспериментов с моделями и признаками необходимо составить таблицу с результатами экспериментов (отобразить модели с гиперпараметрами, результаты метрик, а также сравнить время обучения моделей). 

Опишите лучшее решение и попробуйте сделать выводы, почему решение оказалось лучшим.

In [3]:
import os
import time
import warnings
import numpy as np
import pandas as pd
from datetime import timedelta

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.metrics import roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

In [4]:
import xgboost as xgb
import lightgbm as lgb

In [8]:
X = pd.read_csv('../formatted_data/X_final.csv')
y = pd.read_csv('../formatted_data/y_final.csv')
RANDOM_STATE = 42
target_cols_final = y.columns.tolist()

Decision Tree

In [72]:
results = {}

for cat in target_cols_final:
    mask = y[cat].notna()
    X_cat = X[mask]
    y_cat = y.loc[mask, cat]

    if len(y_cat) == 0:
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X_cat, y_cat, test_size=0.2, random_state=RANDOM_STATE
    )

    dt_model = DecisionTreeClassifier(
        max_depth=20, min_samples_split=5, random_state=RANDOM_STATE
    )

    dt_model.fit(X_train, y_train)

    preds = dt_model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, preds)

    results[cat] = {
        "model": "DecisionTree",
        "params": "max_depth=20, min_samples_split=5",
        "roc_auc": score
    }

results_df_base = pd.DataFrame.from_dict(results, orient="index")
print(results_df_base)

                                model                             params  \
acute_toxicity           DecisionTree  max_depth=20, min_samples_split=5   
carcinogenicity          DecisionTree  max_depth=20, min_samples_split=5   
cardiotoxicity           DecisionTree  max_depth=20, min_samples_split=5   
dermal_toxicity          DecisionTree  max_depth=20, min_samples_split=5   
genotoxicity             DecisionTree  max_depth=20, min_samples_split=5   
hepatotoxicity           DecisionTree  max_depth=20, min_samples_split=5   
ocular_toxicity          DecisionTree  max_depth=20, min_samples_split=5   
oxidative_stress         DecisionTree  max_depth=20, min_samples_split=5   
respiratory_toxicity     DecisionTree  max_depth=20, min_samples_split=5   
neuro_sensory_toxicity   DecisionTree  max_depth=20, min_samples_split=5   
immuno_hematotoxicity    DecisionTree  max_depth=20, min_samples_split=5   
reprod_dev_toxicity      DecisionTree  max_depth=20, min_samples_split=5   
endocrine_me

In [69]:
save_dir = "../models/decision_tree_models/"
os.makedirs(save_dir, exist_ok=True)


param_grid = {
    "max_depth": [5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"]
}

def grid_search_all_categories(X, y, save_dir):
    results_list = []

    for cat in y.columns.tolist():
        mask = y[cat].notna()
        X_cat = X[mask]
        y_cat = y.loc[mask, cat]

        if len(y_cat) == 0:
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X_cat, y_cat, test_size=0.2, random_state=42
        )

        dt = DecisionTreeClassifier(random_state=42)
        grid_search = GridSearchCV(
            dt,
            param_grid,
            cv=3,
            scoring="roc_auc",
            n_jobs=-1
        )

        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        # Предсказание на тесте
        preds = best_model.predict(X_test)

        roc_auc = roc_auc_score(y_test, preds)

        # Сохраняем модель
        model_path = os.path.join(save_dir, f"DecisionTree_{cat}.joblib")
        joblib.dump(best_model, model_path)

        results_list.append({
            "category": cat,
            "best_params": grid_search.best_params_,
            "roc_auc": roc_auc,
            "model_path": model_path
        })

        print(f"{cat}: ROC-AUC={roc_auc:.4f}")

    return pd.DataFrame(results_list)

# Использование
results_df = grid_search_all_categories(X, y, save_dir)
print(results_df)

acute_toxicity: ROC-AUC=0.6862
carcinogenicity: ROC-AUC=0.5609
cardiotoxicity: ROC-AUC=0.5964
dermal_toxicity: ROC-AUC=0.6240
genotoxicity: ROC-AUC=0.7760
hepatotoxicity: ROC-AUC=0.6239
ocular_toxicity: ROC-AUC=0.8195
oxidative_stress: ROC-AUC=0.5751
respiratory_toxicity: ROC-AUC=0.6766
neuro_sensory_toxicity: ROC-AUC=0.6004
immuno_hematotoxicity: ROC-AUC=0.6431
reprod_dev_toxicity: ROC-AUC=0.4901
endocrine_metabolic_tox: ROC-AUC=0.5852
                   category  \
0            acute_toxicity   
1           carcinogenicity   
2            cardiotoxicity   
3           dermal_toxicity   
4              genotoxicity   
5            hepatotoxicity   
6           ocular_toxicity   
7          oxidative_stress   
8      respiratory_toxicity   
9    neuro_sensory_toxicity   
10    immuno_hematotoxicity   
11      reprod_dev_toxicity   
12  endocrine_metabolic_tox   

                                          best_params   roc_auc  \
0   {'criterion': 'gini', 'max_depth': 5, 'min_sam...  0.

Random Forest

In [70]:
save_dir = "../models/forest_models/"
os.makedirs(save_dir, exist_ok=True)

rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "criterion": ["gini"],
    "max_features": ["sqrt"]
}

def grid_search_all_categories_rf(X, y, save_dir):
    results_list = []

    for cat in y.columns.tolist():
        mask = y[cat].notna()
        X_cat = X[mask]
        y_cat = y.loc[mask, cat]

        if len(y_cat) == 0:
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X_cat, y_cat, test_size=0.2, random_state=42
        )

        rf = RandomForestClassifier(random_state=42, n_jobs=-1)
        grid_search = GridSearchCV(
            rf,
            rf_param_grid,
            cv=3,
            scoring="roc_auc",
            n_jobs=-1
        )

        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        if hasattr(best_model, "predict_proba"):
            preds = best_model.predict_proba(X_test)[:,1]
        else:
            preds = best_model.predict(X_test)

        roc_auc = roc_auc_score(y_test, preds)

        model_path = os.path.join(save_dir, f"RandomForest_{cat}.joblib")
        joblib.dump(best_model, model_path)

        results_list.append({
            "category": cat,
            "best_params": grid_search.best_params_,
            "roc_auc": roc_auc
        })

        print(f"{cat}: ROC-AUC={roc_auc:.4f}")

    return pd.DataFrame(results_list)

results_rf = grid_search_all_categories_rf(X, y, save_dir)
print(results_rf)

acute_toxicity: ROC-AUC=0.8410
carcinogenicity: ROC-AUC=0.7315
cardiotoxicity: ROC-AUC=0.9035
dermal_toxicity: ROC-AUC=0.8013
genotoxicity: ROC-AUC=0.9116
hepatotoxicity: ROC-AUC=0.8360
ocular_toxicity: ROC-AUC=0.8961
oxidative_stress: ROC-AUC=0.8053
respiratory_toxicity: ROC-AUC=0.8829
neuro_sensory_toxicity: ROC-AUC=0.8557
immuno_hematotoxicity: ROC-AUC=0.8146
reprod_dev_toxicity: ROC-AUC=0.8817
endocrine_metabolic_tox: ROC-AUC=0.7726
                   category  \
0            acute_toxicity   
1           carcinogenicity   
2            cardiotoxicity   
3           dermal_toxicity   
4              genotoxicity   
5            hepatotoxicity   
6           ocular_toxicity   
7          oxidative_stress   
8      respiratory_toxicity   
9    neuro_sensory_toxicity   
10    immuno_hematotoxicity   
11      reprod_dev_toxicity   
12  endocrine_metabolic_tox   

                                          best_params   roc_auc  
0   {'criterion': 'gini', 'max_depth': 10, 'max_fe...  0.8

Бустинги

In [18]:
lgb_param_grid = {
    "num_leaves": [31, 63],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
    "subsample": [0.8, 1.0]
}
save_dir = "../models/LightGBM/"
os.makedirs(save_dir, exist_ok=True)

def train_models(X, y, save_dir):
    results_list = []

    for cat in y.columns.tolist():
        mask = y[cat].notna()
        X_cat = X[mask]
        y_cat = y.loc[mask, cat]

        if len(y_cat) == 0:  
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X_cat, y_cat, test_size=0.2, random_state=42
        )

        lgb_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1, device='cpu')
        lgb_grid = GridSearchCV(lgb_model, lgb_param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=2)
        lgb_grid.fit(X_train, y_train)
        lgb_best = lgb_grid.best_estimator_
        lgb_preds = lgb_best.predict_proba(X_test)[:,1]
        lgb_roc = roc_auc_score(y_test, lgb_preds)
        lgb_path = os.path.join(save_dir, f"LightGBM_{cat}.joblib")
        joblib.dump(lgb_best, lgb_path)

        results_list.append({
            "category": cat,
            "best_params": lgb_grid.best_params_,
            "roc_auc": lgb_roc
        })

        print(f"{cat}: ROC-AUC={lgb_roc:.4f}")

    return pd.DataFrame(results_list)

results_lgbm = train_models(X, y, save_dir)
print(results_lgbm)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[LightGBM] [Info] Number of positive: 1252, number of negative: 2840
[LightGBM] [Info] Number of positive: 1252, number of negative: 2840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014689 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12620
[LightGBM] [Info] Number of data points in the train set: 4092, number of used features: 78
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.305963 -> initscore=-0.819062
[LightGBM] [Info] Start training from score -0.819062
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022598 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12620
[LightGBM] [Info] Number of data points in the train 

In [19]:
print(results_lgbm)

                   category  \
0            acute_toxicity   
1           carcinogenicity   
2            cardiotoxicity   
3           dermal_toxicity   
4              genotoxicity   
5            hepatotoxicity   
6           ocular_toxicity   
7          oxidative_stress   
8      respiratory_toxicity   
9    neuro_sensory_toxicity   
10    immuno_hematotoxicity   
11      reprod_dev_toxicity   
12  endocrine_metabolic_tox   

                                          best_params   roc_auc  
0   {'learning_rate': 0.05, 'n_estimators': 200, '...  0.861066  
1   {'learning_rate': 0.05, 'n_estimators': 400, '...  0.741842  
2   {'learning_rate': 0.1, 'n_estimators': 400, 'n...  0.915197  
3   {'learning_rate': 0.05, 'n_estimators': 400, '...  0.809863  
4   {'learning_rate': 0.1, 'n_estimators': 200, 'n...  0.917558  
5   {'learning_rate': 0.05, 'n_estimators': 200, '...  0.833305  
6   {'learning_rate': 0.1, 'n_estimators': 200, 'n...  0.901325  
7   {'learning_rate': 0.05, 'n_estima

In [15]:
xgb_param_grid = {
    "max_depth": [6, 10],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}
save_dir = "../models/XGBoost/"
os.makedirs(save_dir, exist_ok=True)


def train_models(X, y, save_dir):

    results_list = []

    for cat in y.columns.tolist():

        mask = y[cat].notna()
        X_cat = X[mask]
        y_cat = y.loc[mask, cat]

        if len(y_cat) == 0:
            continue

        X_train, X_test, y_train, y_test = train_test_split(X_cat,y_cat,test_size=0.2,random_state=42)

        xgb_model = xgb.XGBClassifier(random_state=42,n_jobs=-1,eval_metric="logloss",tree_method="hist")

        xgb_grid = GridSearchCV(xgb_model,xgb_param_grid,cv=3,scoring="roc_auc",n_jobs=-1, verbose=2)

        xgb_grid.fit(X_train, y_train)

        xgb_best = xgb_grid.best_estimator_

        preds = xgb_best.predict_proba(X_test)[:, 1]

        roc = roc_auc_score(y_test, preds)

        model_path = os.path.join(save_dir, f"XGBoost_{cat}.joblib")
        joblib.dump(xgb_best, model_path)

        results_list.append({
            "category": cat,
            "best_params": xgb_grid.best_params_,
            "roc_auc": roc
        })

        print(f"{cat}: ROC-AUC={roc:.4f}")

    return pd.DataFrame(results_list)


results_xgb = train_models(X, y, save_dir)

print(results_xgb)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   2.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   2.3s
[CV] END co

In [16]:
print(results_xgb)

                   category  \
0            acute_toxicity   
1           carcinogenicity   
2            cardiotoxicity   
3           dermal_toxicity   
4              genotoxicity   
5            hepatotoxicity   
6           ocular_toxicity   
7          oxidative_stress   
8      respiratory_toxicity   
9    neuro_sensory_toxicity   
10    immuno_hematotoxicity   
11      reprod_dev_toxicity   
12  endocrine_metabolic_tox   

                                          best_params   roc_auc  
0   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.861911  
1   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.747549  
2   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.921538  
3   {'colsample_bytree': 1.0, 'learning_rate': 0.0...  0.812339  
4   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.921784  
5   {'colsample_bytree': 1.0, 'learning_rate': 0.0...  0.834873  
6   {'colsample_bytree': 1.0, 'learning_rate': 0.1...  0.900618  
7   {'colsample_bytree': 1.0, 'learni

Работа с признаками. Важность 

In [22]:
importance_dict = {}

for cat in y.columns:

    model_path = f"../models/XGBoost/XGBoost_{cat}.joblib"

    if not os.path.exists(model_path):
        continue

    model = joblib.load(model_path)

    importance_dict[cat] = model.feature_importances_

In [ ]:
importance_df = pd.DataFrame(
    importance_dict,
    index=X.columns
)

importance_df["mean_importance"] = importance_df.mean(axis=1)

In [29]:
top50 = importance_df.sort_values(
    "mean_importance",
    ascending=False
).head(50).index

top30 = importance_df.sort_values(
    "mean_importance",
    ascending=False
).head(30).index

X_top50 = X[top50]
X_top30 = X[top30]

print(X.shape)
print(X_top50.shape)
print(X_top30.shape)

(339055, 78)
(339055, 50)
(339055, 30)


In [33]:
xgb_param_grid = {
    "max_depth": [6, 10],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}
save_dir = "../models/XGBoost_50/"
os.makedirs(save_dir, exist_ok=True)


def train_models(X, y, save_dir):

    results_list = []

    for cat in y.columns.tolist():

        mask = y[cat].notna()
        X_cat = X[mask]
        y_cat = y.loc[mask, cat]

        if len(y_cat) == 0:
            continue

        X_train, X_test, y_train, y_test = train_test_split(X_cat,y_cat,test_size=0.2,random_state=42)

        xgb_model = xgb.XGBClassifier(random_state=42,n_jobs=-1,eval_metric="logloss",tree_method="hist")

        xgb_grid = GridSearchCV(xgb_model,xgb_param_grid,cv=3,scoring="roc_auc",n_jobs=-1, verbose=2)

        xgb_grid.fit(X_train, y_train)

        xgb_best = xgb_grid.best_estimator_

        preds = xgb_best.predict_proba(X_test)[:, 1]

        roc = roc_auc_score(y_test, preds)

        model_path = os.path.join(save_dir, f"XGBoost_50_{cat}.joblib")
        joblib.dump(xgb_best, model_path)

        results_list.append({
            "category": cat,
            "best_params": xgb_grid.best_params_,
            "roc_auc": roc
        })

        print(f"{cat}: ROC-AUC={roc:.4f}")

    return pd.DataFrame(results_list)


results_xgb_50 = train_models(X_top50, y, save_dir)


Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   0.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   0.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   0.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   0.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   0.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   0.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   1.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   1.4s
[CV] END co

In [34]:
print(results_xgb_50)

                   category  \
0            acute_toxicity   
1           carcinogenicity   
2            cardiotoxicity   
3           dermal_toxicity   
4              genotoxicity   
5            hepatotoxicity   
6           ocular_toxicity   
7          oxidative_stress   
8      respiratory_toxicity   
9    neuro_sensory_toxicity   
10    immuno_hematotoxicity   
11      reprod_dev_toxicity   
12  endocrine_metabolic_tox   

                                          best_params   roc_auc  
0   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.861270  
1   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.742087  
2   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.919070  
3   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.804298  
4   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.918469  
5   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.828760  
6   {'colsample_bytree': 1.0, 'learning_rate': 0.0...  0.905417  
7   {'colsample_bytree': 0.8, 'learni

In [35]:
xgb_param_grid = {
    "max_depth": [6, 10],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}
save_dir = "../models/XGBoost_30/"
os.makedirs(save_dir, exist_ok=True)


def train_models(X, y, save_dir):

    results_list = []

    for cat in y.columns.tolist():

        mask = y[cat].notna()
        X_cat = X[mask]
        y_cat = y.loc[mask, cat]

        if len(y_cat) == 0:
            continue

        X_train, X_test, y_train, y_test = train_test_split(X_cat,y_cat,test_size=0.2,random_state=42)

        xgb_model = xgb.XGBClassifier(random_state=42,n_jobs=-1,eval_metric="logloss",tree_method="hist")

        xgb_grid = GridSearchCV(xgb_model,xgb_param_grid,cv=3,scoring="roc_auc",n_jobs=-1, verbose=2)

        xgb_grid.fit(X_train, y_train)

        xgb_best = xgb_grid.best_estimator_

        preds = xgb_best.predict_proba(X_test)[:, 1]

        roc = roc_auc_score(y_test, preds)

        model_path = os.path.join(save_dir, f"XGBoost_30_{cat}.joblib")
        joblib.dump(xgb_best, model_path)

        results_list.append({
            "category": cat,
            "best_params": xgb_grid.best_params_,
            "roc_auc": roc
        })

        print(f"{cat}: ROC-AUC={roc:.4f}")

    return pd.DataFrame(results_list)


results_xgb_30 = train_models(X_top30, y, save_dir)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.1s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   2.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   2.2s
[CV] END co

In [36]:
print(results_xgb_30)

                   category  \
0            acute_toxicity   
1           carcinogenicity   
2            cardiotoxicity   
3           dermal_toxicity   
4              genotoxicity   
5            hepatotoxicity   
6           ocular_toxicity   
7          oxidative_stress   
8      respiratory_toxicity   
9    neuro_sensory_toxicity   
10    immuno_hematotoxicity   
11      reprod_dev_toxicity   
12  endocrine_metabolic_tox   

                                          best_params   roc_auc  
0   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.857139  
1   {'colsample_bytree': 0.8, 'learning_rate': 0.1...  0.728396  
2   {'colsample_bytree': 1.0, 'learning_rate': 0.0...  0.912765  
3   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.780945  
4   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.915043  
5   {'colsample_bytree': 1.0, 'learning_rate': 0.0...  0.810103  
6   {'colsample_bytree': 1.0, 'learning_rate': 0.0...  0.898806  
7   {'colsample_bytree': 1.0, 'learni

Кластеризация 

In [37]:
from sklearn.cluster import MiniBatchKMeans

In [46]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [47]:

kmeans = MiniBatchKMeans(
    n_clusters=20,
    random_state=42,
    batch_size=10000
)
clusters = kmeans.fit_predict(X_scaled)


In [49]:
X_cluster = pd.DataFrame(X_scaled, columns=X.columns)
X_cluster["cluster"] = clusters

distances = kmeans.transform(X)

dist_df = pd.DataFrame(
    distances,
    columns=[f"cluster_dist_{i}" for i in range(distances.shape[1])]
)

X_cluster = pd.concat([X_cluster, dist_df], axis=1)

In [50]:
print(X.shape)
print(X_cluster.shape)


(339055, 78)
(339055, 99)


In [51]:
xgb_param_grid = {
    "max_depth": [6, 10],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}
save_dir = "../models/XGBoost_clus/"
os.makedirs(save_dir, exist_ok=True)


def train_models(X, y, save_dir):

    results_list = []

    for cat in y.columns.tolist():

        mask = y[cat].notna()
        X_cat = X[mask]
        y_cat = y.loc[mask, cat]

        if len(y_cat) == 0:
            continue

        X_train, X_test, y_train, y_test = train_test_split(X_cat,y_cat,test_size=0.2,random_state=42)

        xgb_model = xgb.XGBClassifier(random_state=42,n_jobs=-1,eval_metric="logloss",tree_method="hist")

        xgb_grid = GridSearchCV(xgb_model,xgb_param_grid,cv=3,scoring="roc_auc",n_jobs=-1, verbose=2)

        xgb_grid.fit(X_train, y_train)

        xgb_best = xgb_grid.best_estimator_

        preds = xgb_best.predict_proba(X_test)[:, 1]

        roc = roc_auc_score(y_test, preds)

        model_path = os.path.join(save_dir, f"XGBoost_clus_{cat}.joblib")
        joblib.dump(xgb_best, model_path)

        results_list.append({
            "category": cat,
            "best_params": xgb_grid.best_params_,
            "roc_auc": roc
        })

        print(f"{cat}: ROC-AUC={roc:.4f}")

    return pd.DataFrame(results_list)


results_xgb_cluster = train_models(X_cluster, y, save_dir)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   3.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   4.1s
[CV] END co

In [59]:
print(results_xgb_cluster)

                   category  \
0            acute_toxicity   
1           carcinogenicity   
2            cardiotoxicity   
3           dermal_toxicity   
4              genotoxicity   
5            hepatotoxicity   
6           ocular_toxicity   
7          oxidative_stress   
8      respiratory_toxicity   
9    neuro_sensory_toxicity   
10    immuno_hematotoxicity   
11      reprod_dev_toxicity   
12  endocrine_metabolic_tox   

                                          best_params   roc_auc  
0   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.861370  
1   {'colsample_bytree': 0.8, 'learning_rate': 0.1...  0.719573  
2   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.920004  
3   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.811883  
4   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.917272  
5   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.833493  
6   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.900336  
7   {'colsample_bytree': 0.8, 'learni

In [60]:
from sklearn.decomposition import PCA

In [62]:
pca = PCA(n_components=10, random_state=42)
pca_components = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame(pca_components, columns=[f"pca_{i}" for i in range(10)])

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

agg_df = pd.DataFrame({
    "mean_descriptors": X_scaled_df.mean(axis=1),
    "median_descriptors": X_scaled_df.median(axis=1),
    "max_descriptors": X_scaled_df.max(axis=1),
    "min_descriptors": X_scaled_df.min(axis=1),
    "std_descriptors": X_scaled_df.std(axis=1)
})

X_final = pd.concat([X_cluster, pca_df, agg_df], axis=1)
X_final.shape

(339055, 114)

In [63]:
xgb_param_grid = {
    "max_depth": [6, 10],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}
save_dir = "../models/XGBoost_clus/"
os.makedirs(save_dir, exist_ok=True)


def train_models(X, y, save_dir):

    results_list = []

    for cat in y.columns.tolist():

        mask = y[cat].notna()
        X_cat = X[mask]
        y_cat = y.loc[mask, cat]

        if len(y_cat) == 0:
            continue

        X_train, X_test, y_train, y_test = train_test_split(X_cat,y_cat,test_size=0.2,random_state=42)

        xgb_model = xgb.XGBClassifier(random_state=42,n_jobs=-1,eval_metric="logloss",tree_method="hist")

        xgb_grid = GridSearchCV(xgb_model,xgb_param_grid,cv=3,scoring="roc_auc",n_jobs=-1, verbose=2)

        xgb_grid.fit(X_train, y_train)

        xgb_best = xgb_grid.best_estimator_

        preds = xgb_best.predict_proba(X_test)[:, 1]

        roc = roc_auc_score(y_test, preds)

        model_path = os.path.join(save_dir, f"XGBoost_f_{cat}.joblib")
        joblib.dump(xgb_best, model_path)

        results_list.append({
            "category": cat,
            "best_params": xgb_grid.best_params_,
            "roc_auc": roc
        })

        print(f"{cat}: ROC-AUC={roc:.4f}")

    return pd.DataFrame(results_list)


results_xgb_f = train_models(X_final, y, save_dir)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   2.1s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   2.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   2.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   2.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   2.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   2.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   4.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   4.3s
[CV] END co

In [64]:
print(results_xgb_f)

                   category  \
0            acute_toxicity   
1           carcinogenicity   
2            cardiotoxicity   
3           dermal_toxicity   
4              genotoxicity   
5            hepatotoxicity   
6           ocular_toxicity   
7          oxidative_stress   
8      respiratory_toxicity   
9    neuro_sensory_toxicity   
10    immuno_hematotoxicity   
11      reprod_dev_toxicity   
12  endocrine_metabolic_tox   

                                          best_params   roc_auc  
0   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.856720  
1   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.740476  
2   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.918407  
3   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.811036  
4   {'colsample_bytree': 0.8, 'learning_rate': 0.0...  0.918238  
5   {'colsample_bytree': 1.0, 'learning_rate': 0.0...  0.833029  
6   {'colsample_bytree': 1.0, 'learning_rate': 0.0...  0.901028  
7   {'colsample_bytree': 1.0, 'learni

In [80]:
results_rf['model_name'] = 'RF'
results_df['model_name'] = 'DT'
results_df_base['model_name'] = 'DTBase'
results_lgbm['model_name'] = 'LightGBM'
results_xgb['model_name'] = 'XGBoost'
results_xgb_30['model_name'] = 'XGBoost_top_30'
results_xgb_50['model_name'] = 'XGBoost_top_50'
results_xgb_cluster['model_name'] = 'XGBoost_cluster'
results_xgb_f['model_name'] = 'XGBoost_pca'


all_results = pd.concat([
    results_rf,
    results_df,
    results_df_base,
    results_lgbm,
    results_xgb,
    results_xgb_30,
    results_xgb_50,
    results_xgb_cluster,
    results_xgb_f
], ignore_index=True)


pivot_roc = all_results.pivot_table(
    index='category',
    columns='model_name',
    values='roc_auc'
).reset_index()

roc_cols = pivot_roc.columns[1:]  
pivot_roc[roc_cols] = pivot_roc[roc_cols].round(2)

pivot_roc['best_model'] = pivot_roc[roc_cols].idxmax(axis=1)

print(pivot_roc)

model_name                 category    DT  LightGBM    RF  XGBoost  \
0                    acute_toxicity  0.69      0.86  0.84     0.86   
1                   carcinogenicity  0.56      0.74  0.73     0.75   
2                    cardiotoxicity  0.60      0.92  0.90     0.92   
3                   dermal_toxicity  0.62      0.81  0.80     0.81   
4           endocrine_metabolic_tox  0.59      0.76  0.77     0.77   
5                      genotoxicity  0.78      0.92  0.91     0.92   
6                    hepatotoxicity  0.62      0.83  0.84     0.83   
7             immuno_hematotoxicity  0.64      0.82  0.81     0.82   
8            neuro_sensory_toxicity  0.60      0.86  0.86     0.86   
9                   ocular_toxicity  0.82      0.90  0.90     0.90   
10                 oxidative_stress  0.58      0.80  0.81     0.80   
11              reprod_dev_toxicity  0.49      0.83  0.88     0.88   
12             respiratory_toxicity  0.68      0.89  0.88     0.89   

model_name  XGBoost

Выводы. 
1. Бустинги показывают результаты лучше чем деверья и лес(в большинстве) 
2. Для каждой категории токчичности есть своя "лучшая" модель 
3. Для некоторых категорий помогло убрать неважные признаки. immuno_hematotoxicity, neuro_sensory_toxicity, ocular_toxicity - лучше предсказываются на топ 50, reprod_dev_toxicity - на топ 30. 
4. Для respiratory_toxicity небольшой прирост дал kmeans. 